# Incremental Capstone 12 - Deep Learning with TensorFlow and Keras

In this notebook, we build will build and train an Autoencoder using convolutional (Encoder) and transposed convolutional layers (Decoder) for dental X-ray denoising.

## Setup

### Imports

In [ ]:
import os
import re
import zipfile
import urllib.request
from collections import Counter

# Standard library
import sys
from pathlib import Path

# Third-party
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Add src directory to path
sys.path.append(str(Path.cwd().parent))

# Local imports
from src.visualization import plot_image_grid
from src.noise import add_gaussian_noise, add_salt_pepper_noise, add_speckle_noise

### Configuration

In [ ]:
# Force CPU only
tf.config.set_visible_devices([], 'GPU')

### EDA Utility Functions

In [ ]:
data_set_name = ''
target_label = ''

def WrapText(text, max_width=15):
    """Wrap text to multiple lines"""
    words = str(text).split()
    lines = []
    current_line = ""
    
    for word in words:
        if len(current_line + " " + word) <= max_width:
            current_line = (current_line + " " + word).strip()
        else:
            if current_line:
                lines.append(current_line)
            current_line = word
    if current_line:
        lines.append(current_line)
    
    return "\n".join(lines)

def DisplayTable(df_target, table_title=None, max_cell_length=30, show_index=False, 
                 wrap_headers=True, header_wrap_width=15, min_height=3,
                 row_height=0.35, font_size=10):

    if df_target.empty:
        print(f"No data to display{': ' + table_title if table_title else ''}")
        return

    n_rows, n_cols = df_target.shape
    
    # Adjust columns if showing index
    if show_index:
        n_cols += 1

    # Calculate width based on longest column name or cell content
    col_widths = []
    for col in df_target.columns:
        if wrap_headers:
            max_len = max(len(line) for line in WrapText(col, header_wrap_width).split('\n'))
        else:
            max_len = len(str(col))
        for val in df_target[col]:
            val_len = len(f"{val:.2f}" if isinstance(val, float) else str(val))
            max_len = max(max_len, val_len)
        col_widths.append(min(max_len, max_cell_length))
    
    # Dynamic figure width based on content
    fig_width = max(sum(col_widths) * 0.15, n_cols * 2.0)
    
    # Calculate extra height for wrapped headers
    if wrap_headers:
        max_header_lines = max(len(WrapText(col, header_wrap_width).split('\n')) for col in df_target.columns)
    else:
        max_header_lines = 1
    
    fig_height = (n_rows + max_header_lines) * row_height
    if table_title:
        fig_height += 0.4
    
    # Ensure minimum height for small tables
    fig_height = max(fig_height, min_height)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Format floats and truncate long text
    cell_text = []
    for idx, row in zip(df_target.index, df_target.values):
        new_row = []
        
        # Add index as first column if show_index
        if show_index:
            s = str(idx)
            if len(s) > max_cell_length:
                s = s[:max_cell_length - 3] + '...'
            new_row.append(s)
        
        for val in row:
            if isinstance(val, (float)) and not isinstance(val, bool):
                new_row.append(f"{val:.2f}")
            else:
                s = str(val)
                if len(s) > max_cell_length:
                    s = s[:max_cell_length - 3] + '...'
                new_row.append(s)
        cell_text.append(new_row)

    # Build column labels (with wrapping)
    if show_index:
        col_labels = [df_target.index.name or '']
        if wrap_headers:
            col_labels += [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels += list(df_target.columns)
    else:
        if wrap_headers:
            col_labels = [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels = list(df_target.columns)

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='upper center',
        bbox=[0, 0, 1, 1]
    )

    # Bold column headers and set header background
    for col in range(n_cols):
        table[(0, col)].set_facecolor('#4a90d9')
        table[(0, col)].set_text_props(color='white', fontweight='bold')

    # Style data cells
    for row in range(1, n_rows + 1):
        for col in range(n_cols):
            if show_index and col == 0:
                table[(row, col)].set_facecolor('#b8d4e8')
                table[(row, col)].set_text_props(fontweight='bold')
            else:
                table[(row, col)].set_facecolor('#d4e6f1')

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.auto_set_column_width(col=list(range(n_cols)))

    if table_title is not None:
        fig.suptitle(table_title, fontweight='bold', fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95] if table_title else [0, 0, 1, 1])

    plt.show()
    plt.close(fig)
    print()
    print()
    
def PrintDataFrameStatistics(df_target, display_table=True):
    # Print Stats
    print("***********************************")
    print("Description Stats")
    print("***********************************")
    print()

    # Capture for Table plot
    df_stats = df_target.describe(include='all').T.reset_index()
    df_stats.rename(columns={'index': 'Feature'}, inplace=True)
    print(df_stats)
    print()

    # Print df Column Info
    print("***********************************")
    print("Basic Info of imported data set")
    print("***********************************")
    print()

    df_info = pd.DataFrame({
                            'Feature': df_target.columns,
                            'Non-Null Count': df_target.notna().sum().values,
                            'Null Count': df_target.isna().sum().values,
                            'Dtype': df_target.dtypes.values
                            }).reset_index(drop=True)

    print(f'Dataset Shape:{df_target.shape}')
    df_shape = pd.DataFrame({'Rows': [df_target.shape[0]], 'Columns': [df_target.shape[1]]})
    print()

    print('Do we have any features with null values?:')
    print(df_target.isnull().any().any())
    print()

    print('Do we have any features empty strings?:')
    print((df_target == "").any())
    print()

    print('Feature Columns with that have null values:')
    print(df_target.isnull().sum()[df_target.isnull().sum() > 0])

    # Capture for Table plot
    cols_to_plot = df_target.select_dtypes(exclude=['number']).columns
    missing = df_target[cols_to_plot].isnull().sum()
    missing = missing[missing > 0]
    df_missing = pd.DataFrame({
                                'Feature': missing.index,
                                'Missing Count': missing.values,
                                'Missing %': (missing.values / len(df_target) * 100).round(2)
                                }).reset_index(drop=True)
    print()

    print('Do we have any features with nan values?:')

    cols_to_plot = df_target.select_dtypes(include=['number']).columns
    nan_vals = df_target[cols_to_plot].isna().sum()
    nan_vals = nan_vals[nan_vals > 0]
    # Capture for Table plot
    df_nan = pd.DataFrame({
                            'Feature': nan_vals.index,
                            'NaN Count': nan_vals.values,
                            'NaN %': (nan_vals.values / len(df_target) * 100).round(2)
                        }).reset_index(drop=True)
    print(df_target.isna().any().any())

    # Sum up the number of missing features per row
    missing_per_row = df_target.isna().sum(axis=1)  # count missing per row
    missing_counts = missing_per_row.value_counts().sort_index()  # count rows for each missing count

    df_missingfeature_rowcounts = pd.DataFrame({
        'Missing Features': missing_counts.index,
        'Row Count': missing_counts.values
    })

    print("***********************************")
    print("First 20 rows of Data")
    print("***********************************")
    print()
    print(df_target.head(20))
    print()

    print("***********************************")
    print("First 20 rows of Random Sample Data")
    print("***********************************")
    print()
    df_randomsample = df_target.sample(n=20)
    print(df_target.sample(20))
    print()

    #
    # Display Results in Pretty Tables
    #
    if display_table == True:
        print('Display Analysis Results in Tables')
        DisplayTable(df_stats, f'{data_set_name} Description Statistics')
        print()
        DisplayTable(df_info, f'{data_set_name} Basic Information')
        print()
        DisplayTable(df_shape, f'{data_set_name} Dataset Shape')
        print()
        DisplayTable(df_missing, f'{data_set_name} Missing Categorical (String) Data')
        print()
        DisplayTable(df_nan, f'{data_set_name} Missing Numeric Data')
        print()
        DisplayTable(df_missingfeature_rowcounts, f'{data_set_name} Summary of Missing Feature Row Counts')
        print()    
        DisplayTable(df_randomsample, f'{data_set_name} Random Data Sample')
        
    print()

## **1. Data preparation** ##

### 1.1 Load
Load the Dental Data using numpy and save test/train split data

In [ ]:
# Some issues figuring out where the actual data lives in the folder structure...
# import os
# print(os.getcwd())
# os.listdir('data/')

# Load the .npz file
data = np.load('../data/DENTAL_1.npz')

# See what arrays are inside
print(data.files)  # e.g., ['x_train', 'y_train', 'x_test', 'y_test']

# Access individual arrays by key
x_train = data['x_train']
y_train = data['y_train']
x_test = data['x_test']
y_test = data['y_test']

# Verify shapes
print(f'Train Shape: {x_train.shape}, {y_train.shape}')
print(f'Test Shape: {x_test.shape}, {y_test.shape}')

# Print sample data array to see what values look like
print(x_train[0][:3, :3])
print()

# Verify that incoming data is normalized between [0..1]
print(f'Train Data Min:{x_train.min()} Max:{x_train.max()}')

### 1.2 Preprocess The Data ###
Visualize a random sample of the incoming data set (from Test split)

In [ ]:
# Generate random indices
random_train_indices = np.random.choice(len(x_train), 10, replace=False)
random_test_indices = np.random.choice(len(x_test), 10, replace=False)

# Plot 10 random images from incoming dataset
def PlotRandomImageSamples(target, random_indices, plot_title):

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(target[random_indices[i]], cmap='gray')
        ax.set_title(f'Index {random_indices[i]}')
        ax.axis('off')
        
    plt.suptitle(f'10 Random {plot_title} Images', fontweight='bold')
    plt.tight_layout()
    plt.show()


# Print sample data array to see what values look like
print(x_train[0][:3, :3])
print()

# Verify that incoming data is normalized between [0..1]
print(f'Train Data Min:{x_train.min()} Max:{x_train.max()}')

PlotRandomImageSamples(x_train, random_train_indices, 'x_train Clean Images')
print()
PlotRandomImageSamples(x_test, random_test_indices, 'x_test Clean Images')

### Add Random Noise to Test and Train Data ###
The incoming dataset is clean dental data (without noise).</br>
The purpose of this exercise is to train an **Autoencoder Model** that will recognize and identify noisy data.</br>
The only way to do that is to take our incoming "**clean**" data, add randomize noise to and use it to train the model.</br>

#### Add Gaussian Noise to incoming "**clean**" dataset.
The add_gaussian_noise() function already clips the data between [0,1] so there is no need to any additional normalization.


In [ ]:
# Direct from George's src\noise.py file
#
def add_gaussian_noise(images, noise_factor=0.3):
    """
    Add Gaussian (normal) noise to images.
    
    Args:
        images: Clean images (numpy array)
        noise_factor: Standard deviation of noise (higher = more noise)
    
    Returns:
        Noisy images clipped to [0, 1]
    """
    noise = np.random.normal(loc=0.0, scale=noise_factor, size=images.shape)
    noisy_images = images + noise
    return np.clip(noisy_images, 0.0, 1.0)

# Add noise to test/train data
x_train_noisy = add_gaussian_noise(x_train)
x_test_noisy = add_gaussian_noise(x_test)

# Plot same random images but with added noise
PlotRandomImageSamples(x_train_noisy, random_train_indices, 'x_train Noisy Images')
print()
PlotRandomImageSamples(x_test_noisy, random_test_indices, 'x_test Clean Images')

# Let's also print out a sample of the altered array data

#### Based on examining the random sample of clean/noisy images, we can verify that the images have been successfully noisyied ####

## **2. Build and Train The Model** ##

### Global Things to Tune ###
Hyperparameters to play with for training the model.

In [ ]:
epochs = 20
batch_size = 32

### Build the AutoEncoder ###
The AutoEncoder starts with the initial convolutional/maxpooling layers from the class demo.
Once there is a working architecture, I can start playing with altering the layers with the model

In [ ]:
# The basis of this was taken from 02-denoising notebook (Class Assignment)
#
def build_denoising_autoencoder(input_shape=(64, 64, 3), latent_dim=128):

    # Encoder
    encoder_input = keras.Input(shape=input_shape)
    
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(encoder_input)
    x = layers.MaxPooling2D(2, padding='same')(x)  # 32x32
    
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)  # 16x16
    
    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)  # 8x8
    
    x = layers.Conv2D(512, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)  # 4x4
    
    # Bottleneck
    x = layers.Flatten()(x)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    
    # Decoder
    x = layers.Dense(16 * 16 * 512, activation='relu')(latent)
    x = layers.Reshape((16, 16, 512))(x)
    
    x = layers.Conv2DTranspose(512, 3, activation='relu', strides=2, padding='same')(x)  # 8x8
    x = layers.Conv2DTranspose(256, 3, activation='relu', strides=2, padding='same')(x)  # 16x16
    x = layers.Conv2DTranspose(128, 3, activation='relu', strides=2, padding='same')(x)  # 32x32
    x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)   # 64x64
    
    decoder_output = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x)
    
    # Build model
    autoencoder = keras.Model(encoder_input, decoder_output, name='denoising_autoencoder')
    
    # Compile
    autoencoder.compile(
                        optimizer='adam',
                        loss='mse',
                        metrics=['mae']
                    )
    
    return autoencoder


# Build model
model = build_denoising_autoencoder(input_shape=(256, 256, 3), 
                                    latent_dim=128)
model.summary()

### Train the AutoEncoder Model ###

In [ ]:
# Train the model
# Input: noisy images, Target: clean images
history = model.fit(
                    x_train_noisy,  # Noisy input
                    x_train,        # Clean target
                    validation_data=(x_test_noisy, x_test),
                    epochs=epochs,
                    batch_size=batch_size,
                    shuffle=True,
                    verbose=1
                )

### Examine/Plot the Training History ###

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# Loss
axes[0].set_title('Training Loss')
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

# MAE
axes[1].set_title('Mean Absolute Error')
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

### Examine Random Image Samples After Training ### 
Let's see what original sample images look like after training:
- Clean Image --> Noisy Image --> Trained Image

In [ ]:
reconstructed_train_images = model.predict(x_train_noisy)

def PlotProcessedImageComparisons(clean, noisy, reconstructed, random_indices, plot_title):

    fig, axes = plt.subplots(3, 10, figsize=(25, 8))

    for i, idx in enumerate(random_indices):
        
        axes[0, i].imshow(clean[idx], cmap='gray')
        axes[0, i].set_title(f'Clean {idx}')
        axes[0, i].axis('off')

        axes[1, i].imshow(noisy[idx], cmap='gray')
        axes[1, i].set_title(f'Noisy {idx}')
        axes[1, i].axis('off')

        axes[2, i].imshow(reconstructed[idx], cmap='gray')
        axes[2, i].set_title(f'Denoised {idx}')
        axes[2, i].axis('off')

    plt.suptitle(f'Clean vs Noisy vs Denoised ({plot_title})', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print()


PlotProcessedImageComparisons(x_train, 
                              x_train_noisy, 
                              reconstructed_train_images, 
                              random_train_indices,
                              'Training Dataset')    

### Run The AutoEncoder Model Against The Test Dataset ###

In [ ]:
# Denoise test images
reconstructed_test_images = model.predict(x_test_noisy, verbose=0)
print()

PlotProcessedImageComparisons(x_test, 
                              x_test_noisy, 
                              reconstructed_test_images, 
                              random_test_indices, 
                              'Test Dataset') 
print()

### Examine Random Image Samples After Training ### 
Let's see what original sample images look like after training:
- Clean Image --> Noisy Image --> Trained Image

##### **Punctuation Examination Results** #####
- ~17% ( 12000/71044) of the ratings have missing punctuation (at end of review)

##### Spelling Analysis #####

##### **Spelling Analysis Results** #####
- Lower ratings have a slightly higher average number of spelling errors

#### Rating Review Analysis ####

##### **Review Length Analysis** #####
- Negative reviews tend to be longer than positive reviews.

## 2. **Preprocess the Data** ##
#### Impute Missing Data ####

#### Label Encoding ####
- review.doRecommend is a categorical value (True/False) and should be mapped into numeric (integer) values
    This will be useful when we compare the end predictions against the intended/actual labels        

In [ ]:
df_grammarAndReviews['reviews.doRecommend'] = df_grammarAndReviews['reviews.doRecommend'].map(
                                                                                                {'True': 1, 'False': 0, True: 1, False: 0}
                                                                                            ).astype('int8')

#### **An Afterthought Regarding Padding** ####
- Padding the review text at this point is an unnecessary step.</br>
    What really matters is padding the review sequences generated by tokenization (occurs further in the processing)</br>
    Since I already did the work, I left it in for posterity's sake, and now I know how to do it.

##### **Count Plot Utility Functions** #####        

## **3.0 Model Creation and Training** ##
#### Build The Model ####
- We are solving a **Binary Classification** Problem:
    - Do Recommend == 1
    - Do Not Recommend == 0 </br>

For **Binary Classification** we use:
- **Sigmoid** as the Activation Function on the Dense (last) layer of the model (b/w 0 and 1)
- **binary_crossentropy** as the loss function on the model

##### **Training and Evaluation Utility Functions** ####

In [ ]:
########################################
##
## RNN Model Template from class demo
##
#######################################
# Model parameters
# hidden_dim = 8 # RNN hidden state size

# model = Sequential([
#     SimpleRNN(hidden_dim, input_shape=(seq_length, vocab_size)),
#     Dense(vocab_size, activation='softmax')
# ])



###############################################
# We're solving Binary Classification problem
###############################################

embedding_dim = 128

def CreateRNNModel(dropout_rate):

    model = Sequential([

        # Embedding - convert 2D sequences into 3D vectors [required by SimpleRNN(), LTSM()]
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        
        # Block 1: Convolutional
        # Text is 1 dimensional sequence - so use 1D function layers
        # relu activation - 
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(dropout_rate),
        
        # Block 2: Recurrent
        SimpleRNN(64, return_sequences=False),
        Dropout(dropout_rate),
        
        # Block 3: Dense
        Dense(32, activation='relu'),
        Dropout(dropout_rate),

        # Binary classification: doRecommend 0 or 1
        Dense(1, activation='sigmoid')  
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()
    return model

def CreateLTSMModel(dropout_rate):
    model = Sequential([

        # Embedding - convert 2D sequences into 3D vectors [required by SimpleRNN(), LTSM()]
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        
        # Block 1: Convolutional
        # Text is 1 dimensional sequence - so use 1D function layers
        # relu activation - 
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(dropout_rate),
        
        # Block 2: Recurrent
        LSTM(64, return_sequences=False),
        Dropout(dropout_rate),
        
        # Block 3: Dense
        Dense(32, activation='relu'),
        Dropout(dropout_rate),

        # Binary classification: doRecommend 0 or 1
        Dense(1, activation='sigmoid')  
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()
    return model


#### Train the Model ####

In [ ]:
# Train the model
def TrainTheModel(model, dropout_rate, epochs):

    sequential_history = model.fit(
                                    X_train, 
                                    y_train,
                                    epochs=epochs,  # 10
                                    batch_size=32,
                                    validation_split=0.2
                                )

    print(f'dropout_rate: {dropout_rate}  loss: {sequential_history.history["loss"]}')
    print(f'dropout_rate: {dropout_rate}  accuracy: {sequential_history.history["accuracy"]}')
    print(f'dropout_rate: {dropout_rate}  val_loss: {sequential_history.history["val_loss"]}')
    print(f'dropout_rate: {dropout_rate}  val_accuracy: {sequential_history.history["val_accuracy"]}')
    return sequential_history

#### Plot the Learning Curves ####

In [ ]:
def PlotLearningCurves(history, plot_title):

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].set_title(f'{plot_title}: loss')
    axes[0].plot(history['loss'], label='Training')
    axes[0].plot(history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (binary crossentropy)')
    axes[0].legend(loc='best')

    axes[1].set_title(f'{plot_title}: accuracy')
    axes[1].plot(history['accuracy'], label='Training')
    axes[1].plot(history['val_accuracy'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(loc='best')

    plt.tight_layout()
    plt.show()

##### **CNN-LSTM - Loss and Accuracy Analysis** ####

#### Test Set Evaluation ####

In [ ]:
def EvaluateModel(model, dropout_rate):
    # Evaluate against test set
    test_loss, test_accuracy = model.evaluate(X_test, y_test)
    print(f'dropout_rate: {dropout_rate}  Test Loss: {test_loss:.4f}')
    print(f'dropout_rate: {dropout_rate}  Test Accuracy: {test_accuracy:.4f}')


#### Confusion Matrix Plots ####

In [ ]:
def PlotConfusionMatrix(model, X_test, y_test, plot_title):

    y_probs = model.predict(X_test).flatten()
    y_pred = (y_probs >= 0.5).astype(int)

    print('----------------------------------------------------------')
    print(plot_title)
    print()
    print(classification_report(y_test, y_pred, target_names=['Not Recommend', 'Recommend']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}')
    print('----------------------------------------------------------')

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Recommend', 'Recommend'])
    disp.plot(cmap='Blues')
    plt.title(plot_title)
    plt.show()

#### Train/Test Wrapper Methods ####

#### Run The Models for RNN and LTSM for epoch/dropout_rate Combinations ####

#### RNN-CNN and LTSM Analysis ####
##### Loss, Accuracy and ConfusionMatrix Analysis Across Dropout Rates #####
- **RNN-CNN**: 
    - Dropout Rate: 0.2
        - Learning Curves: training/validation track closely, no signficant gap. Accuracy peaks ~91%. Not overfitting.
        - ConfusionMatrix: Does well on correct Recommended, but fails on Not Recommended
    - Dropout Rate: 0.3
        - Learning Curves: Loss increases after 8 epochs, while accuracy increases. Overfitting.
        - ConfusionMatrix: Same issues
    - Dropout Rate: 0.4
        - Learning Curves: Spikes in validation. Huge decrease and upward bounce in Validation Accuracy at 8 epochs. (too large of a dropout_rate??)
        - ConfusionMatrix: Same issues
    - Dropout Rate: 0.5
        - Learning Curves: Training/Validation track close with no significant gaps. Loss spike at 7 epochs. Slow learning (large dropout_rate?). Not overfitting
        - ConfusionMatrix: Same issues   
- **LTSM-CNN**: 
    - Dropout Rate: 0.2
        - Learning Curves: Overfitting. Training loss decrease while validation loss increases. Training accuracy high while validation low and fluctuates (large gap).
        - ConfusionMatrix: Performs better than RNN-CNN. More false Recommendeds and more postive Not Recommendeds
    - Dropout Rate: 0.3
        - Learning Curve: Similar to 0.2
        - ConfusionMatrix: Performs slightly worse for false Not Recommendeds
    - Dropout Rate: 0.4
        - Learning Curves: Similar to 0.2
        - ConfusionMatrix: Performs better for positive Not Recommendeds
    - Dropout Rate: 0.5
        - Learning Curves: Severe overfitting - large gaps. Validation looks unstable/jagged
        - ConfusionMatrix: Performs worse than 0.4                        
</br>
##### Display Top Sorted Metrics and Plot Metric Comparisons Between **RNN-CNN** and **LSTM-CNN** #####


#### Metrics Analysis ####
- The dataset is definitely unbalanced - as evidenced by looking at ConfusionMatrices - "**Do Recommends**"</br>
With more time I could have played with class weighting to see if that helped things out.
- **F1 Score** is the metric of choice because this is a **Classification Problem** with **imbalanced classes**.

#### **Best Overall Performing Model/Dropout Rate Combination** ####
| model_name | dropout_rate | epochs | test_loss | test_accuracy | precision | recall | f1 | roc_auc |
|------------|--------------|--------|-----------|--------------|----------|--------|------|---------|
| CNN-LSTM   | 0.2          | 10     | 0.298357  | 0.931841     | 0.960210 | 0.964972 | 0.962585 | 0.885313 |

</br>

## **CNN-LSTM** with **dropout_rate=0.2**
Is the best model/dropout rate combination. 
- Highest F1 Score
- Reasonably high ROC-AUC
- Lowest loss
- Reasonably high accuracy without overfitting

##### **Final Thoughts and Observations** #####
First thing I learned with this assignment is that there is a lot of information into the target participants that you can glean with some </br>
simple straightforward analysis of the text itself (spelling, punctionation, length, et al). There is also a fair amount of preprocessing</br>
required to get the text into something that's normalized and easily consumed by models (tokenizing - models like working with numbers)v
With more time, it would have been fun to look at GRU and play with hyperparameter tuning to get better performance. </br>
We could have even factored some of the other features into the model, but hat was not the main goal of this exercise. </br>
The main goal was to see how we could combine RNN and CNN to build a predictive model based on text fields.</br>
</br>
.. **and finally**, I wish I had my new GPU based machine (either *Mr. Windows Desktop* or *Mr. Mc-Linux-Beast*) to make the processing of the assignments more efficient.
